# 03 - Concept and Relationship Extraction Demo

Demonstrate extraction of concepts (NER, noun-phrases, TF-IDF) and relationships (co-occurrence, dependency parsing) from combined policy + Yelp data.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.ingest.pdf_reader import read_policy_pdf, pages_to_document
from src.ingest.yelp_reader import load_yelp_reviews
from src.preprocessing.cleaner import clean_text
from src.preprocessing.tokeniser import SpacyTokeniser
from src.extraction.concept_extractor import ConceptExtractor
from src.extraction.relationship_extractor import RelationshipExtractor

## 1. Prepare Combined Data (Policy + Yelp)

In [ ]:
tokeniser = SpacyTokeniser()
documents = []

# Policy document
pages = read_policy_pdf()
full_text = pages_to_document(pages)
cleaned_policy = clean_text(full_text, source='policy')
documents.append(tokeniser.process(cleaned_policy, doc_id='policy', source='policy'))
print(f'Policy: {len(documents[0].sentences)} sentences')

# Yelp reviews (small sample for demo)
df = load_yelp_reviews(category_filter='Restaurants', sample_n=50)
yelp_texts = [(row['review_id'], row['text']) for _, row in df.iterrows()]
cleaned_yelp = [(did, clean_text(t, source='yelp')) for did, t in yelp_texts]
cleaned_yelp = [(did, t) for did, t in cleaned_yelp if t.strip()]
documents.extend(tokeniser.process_batch(cleaned_yelp, source='yelp'))

total_sents = sum(len(d.sentences) for d in documents)
print(f'Combined: {len(documents)} documents, {total_sents} sentences')

## 2. Concept Extraction

In [ ]:
extractor = ConceptExtractor(use_ner=True, use_noun_phrases=True, use_tfidf=True)
concepts = extractor.extract(documents)

print(f'Total concepts: {len(concepts)}')
print(f'  Entities: {sum(1 for c in concepts if c.type == "entity")}')
print(f'  Noun phrases: {sum(1 for c in concepts if c.type == "noun_phrase")}')
print(f'  TF-IDF keywords: {sum(1 for c in concepts if c.type == "keyword")}')

print(f'\nBy source origin:')
print(f'  Policy-only: {sum(1 for c in concepts if c.source_type == "policy")}')
print(f'  Yelp-only:   {sum(1 for c in concepts if c.source_type == "yelp")}')
print(f'  Both:        {sum(1 for c in concepts if c.source_type == "both")}')

print('\nTop 15 concepts by frequency:')
for c in concepts[:15]:
    print(f'  [{c.type:12s}] {c.label} (freq={c.frequency}, src={c.source_type})')

## 3. Relationship Extraction

In [ ]:
rel_extractor = RelationshipExtractor(use_cooccurrence=True, use_dependency=True)
relationships = rel_extractor.extract(documents, concepts)

from collections import Counter
type_counts = Counter(r.type for r in relationships)
print(f'Total relationships: {len(relationships)}')
print(f'By type: {dict(type_counts)}')

print('\nTop 15 relationships by weight:')
for r in relationships[:15]:
    print(f'  {r.source_id} --[{r.type}, w={r.weight}]--> {r.target_id}')